In [ ]:
### preprocess.py ###
import pandas as pd
import networkx as nx
import numpy as np
import pickle
import os

def build_supply_chain_network(data_folder='.'):
    """
    该函数读取CSV格式的业务数据，基于销量分布反向构建供应链网络。
    逻辑：
    1. 初始节点集合 = 销量分布中的 (Location, SKU)。
    2. 遍历集合：
       - 如果节点通过补货获取，将上游 (SourceLocation, SKU) 加入集合。
       - 如果节点是IDC且有BOM结构，将原材料 (IDC, Material) 加入集合。
    3. 仅为集合中的有效节点构建图和边。
    """
    print("--- 开始数据预处理 ---")

    # --- 1. 加载所有CSV数据 ---
    try:
        df_locations = pd.read_csv(os.path.join(data_folder, '仓库主数据.csv'))
        df_products = pd.read_csv(os.path.join(data_folder, '产品主数据.csv'))
        df_bom = pd.read_csv(os.path.join(data_folder, '生产物料清单.csv'))
        df_idc_cdc = pd.read_csv(os.path.join(data_folder, 'IDC-CDC补货参数.csv'))
        df_cdc_rdc = pd.read_csv(os.path.join(data_folder, 'CDC-RDC补货参数.csv'))
        df_sales = pd.read_csv(os.path.join(data_folder, '销量分布.csv'))
    except FileNotFoundError as e:
        print(f"错误：文件未找到。请确保所有CSV文件都在指定的路径中: '{data_folder}'")
        print(f"具体错误信息: {e}")
        return None
        
    print("所有CSV文件加载成功。")

    # --- 2. 整合SKU主数据 ---
    print("正在整合SKU主数据...")
    df_fg_costs = df_products[['SKUCode', 'ProcurementCost']].rename(columns={'ProcurementCost': 'Cost'})
    df_rm_costs = df_bom[['MaterialCode', 'UnitOfPrice']].rename(columns={'MaterialCode': 'SKUCode', 'UnitOfPrice': 'Cost'})
    df_rm_costs = df_rm_costs.drop_duplicates(subset=['SKUCode']).reset_index(drop=True)
    df_sku_master = pd.concat([df_fg_costs, df_rm_costs]).drop_duplicates(subset=['SKUCode']).reset_index(drop=True)

    # 识别各类集合
    fg_skus = set(df_products['SKUCode'].unique())
    rm_skus = set(df_bom['MaterialCode'].unique())
    customer_facing_locs = set(df_locations[df_locations['FacingCustomer'] == True]['LocationCode'].unique())
    
    # IDCCN01 是唯一的生产/原材料存放节点
    idc_loc = 'IDCCN01' 

    # --- 2.5 核心逻辑修改：基于需求反向回溯有效节点 ---
    print("正在根据销量分布回溯构建供应链网络...")

    # 2.5.1 准备查找表以加速回溯
    # 补货映射: (DestLoc, SKU) -> SourceLoc
    replenishment_map = {}
    df_replenishment_all = pd.concat([df_idc_cdc, df_cdc_rdc])
    for _, row in df_replenishment_all.iterrows():
        replenishment_map[(row['DestLocationCode'], row['SKUCode'])] = row['SourceLocationCode']
    
    # BOM 映射: FG -> List of [RM]
    bom_map = {}
    for _, row in df_bom.iterrows():
        fg = row['SKUCode']
        rm = row['MaterialCode']
        if fg not in bom_map:
            bom_map[fg] = []
        bom_map[fg].append(rm)

    # 2.5.2 初始化：从销量数据开始
    # valid_nodes 存储元组: (LocationCode, SKUCode)
    initial_demand = df_sales[['LocationCode', 'SKUCode']].drop_duplicates().values.tolist()
    valid_nodes = set()
    queue = []

    for loc, sku in initial_demand:
        node_pair = (loc, sku)
        if node_pair not in valid_nodes:
            valid_nodes.add(node_pair)
            queue.append(node_pair)
    
    print(f"初始销量分布涉及 {len(valid_nodes)} 个节点。")

    # 2.5.3 广度优先搜索 (BFS) 回溯上游
    while queue:
        curr_loc, curr_sku = queue.pop(0)

        # 逻辑 A: 检查补货来源 (Replenishment)
        if (curr_loc, curr_sku) in replenishment_map:
            source_loc = replenishment_map[(curr_loc, curr_sku)]
            upstream_node = (source_loc, curr_sku)
            
            if upstream_node not in valid_nodes:
                valid_nodes.add(upstream_node)
                queue.append(upstream_node)

        # 逻辑 B: 检查生产需求 (Production / BOM)
        if curr_loc == idc_loc and curr_sku in bom_map:
            rm_list = bom_map[curr_sku]
            for rm_sku in rm_list:
                rm_node = (idc_loc, rm_sku)
                
                if rm_node not in valid_nodes:
                    valid_nodes.add(rm_node)
                    queue.append(rm_node)

    print(f"回溯完成，最终网络包含 {len(valid_nodes)} 个必要节点 (SKU-Location 组合)。")

    # --- 3. 初始化网络并创建节点 ---
    G = nx.DiGraph()
    node_map = {} # (loc, sku) -> node_id
    node_counter = 0

    print("正在创建网络节点...")
    for loc, sku in sorted(list(valid_nodes)):
        node_name = f"{loc}_{sku}"
        node_map[(loc, sku)] = node_counter
        
        is_facing = loc in customer_facing_locs
        
        if sku in fg_skus:
            sku_type = 'FG'
        elif sku in rm_skus:
            sku_type = 'RM'
            
        G.add_node(node_counter, name=node_name, location=loc, sku=sku, 
                   is_customer_facing=is_facing,
                   sku_type=sku_type)
        node_counter += 1

    # --- 4. 计算并设置节点属性 ---
    print("正在设置节点属性 (成本)...")
    loc_info_map = df_locations.set_index('LocationCode')[['HoldingCostRate', 'LocationType']].to_dict('index')
    sku_cost_map = df_sku_master.set_index('SKUCode')['Cost'].to_dict()

    nx.set_node_attributes(G, name='holdcost', values=0.0)
    
    for node_id in G.nodes():
        node_data = G.nodes[node_id]
        loc = node_data['location']
        sku = node_data['sku']
        
        procurement_cost = sku_cost_map.get(sku, 0.0)
        holding_rate = 0.0
        if loc in loc_info_map:
            holding_rate = loc_info_map[loc]['HoldingCostRate']
            
        holding_cost = procurement_cost * (1 + holding_rate)
        
        nx.set_node_attributes(G, {node_id: {'holdcost': round(holding_cost, 4)}})

    # --- 5. 构建网络边 ---
    print("正在构建网络拓扑 (边)...")
    
    # 5.1 生产边 (BOM)
    for _, row in df_bom.iterrows():
        fg_sku, rm_sku = row['SKUCode'], row['MaterialCode']
        if (idc_loc, rm_sku) in node_map and (idc_loc, fg_sku) in node_map:
            u = node_map[(idc_loc, rm_sku)]
            v = node_map[(idc_loc, fg_sku)]
            G.add_edge(v, u, weight=row['MaterialQty'], type='production')

    # 5.2 补货边
    replenishment_links = pd.concat([df_idc_cdc, df_cdc_rdc])
    for _, row in replenishment_links.iterrows():
        source_loc, dest_loc, sku = row['SourceLocationCode'], row['DestLocationCode'], row['SKUCode']
        
        if (source_loc, sku) in node_map and (dest_loc, sku) in node_map:
            u = node_map[(source_loc, sku)]
            v = node_map[(dest_loc, sku)]

            G.add_edge(v, u, weight=1.0, type='replenishment', min_lot_size=row.get('MinLotSizes', 0))
            
            node_attrs = {
                'leadtime': row.get('LeadTime', 1),
                'service_level': row.get('ServiceLevel', 0.95),
                'replenishment_cycle': row.get('ReplenishmentCycle', 1)
            }
            nx.set_node_attributes(G, {v: node_attrs})

    # --- 5.5 为缺少补货参数的节点设置默认值 ---
    print("为缺少属性的节点设置默认参数...")
    
    for node_id, attrs in G.nodes(data=True):
        loc = attrs['location']
        loc_type = loc_info_map.get(loc, {}).get('LocationType', '')
        
        defaults_to_set = {}
        
        if loc_type == 'CDC':
            if 'leadtime' not in attrs: defaults_to_set['leadtime'] = 7
            if 'replenishment_cycle' not in attrs: defaults_to_set['replenishment_cycle'] = 30
            if 'service_level' not in attrs: defaults_to_set['service_level'] = 0.98
        elif loc_type == 'RDC':
            if 'leadtime' not in attrs: defaults_to_set['leadtime'] = 7
            if 'replenishment_cycle' not in attrs: defaults_to_set['replenishment_cycle'] = 7
            if 'service_level' not in attrs: defaults_to_set['service_level'] = 0.95
        elif loc_type == 'IDC':
            if 'leadtime' not in attrs: defaults_to_set['leadtime'] = 1
            if 'replenishment_cycle' not in attrs: defaults_to_set['replenishment_cycle'] = 1
            if 'service_level' not in attrs: defaults_to_set['service_level'] = 0.98
             
        if defaults_to_set:
            nx.set_node_attributes(G, {node_id: defaults_to_set})

    # --- 6. 清理和保存 ---
    G = nx.convert_node_labels_to_integers(G, first_label=0, ordering='default')
    
    print(f"最终网络包含 {G.number_of_nodes()} 个节点和 {G.number_of_edges()} 条边。")

    output_filename = 'supply_chain_network.pkl'
    output_path = os.path.join(data_folder, output_filename)
    with open(output_path, 'wb') as f:
        pickle.dump(G, f)
    print(f"--- 预处理完成！网络模型已保存至 {output_filename} ---")
    
    return G

if __name__ == "__main__":
    data_folder_name = 'data'
    current_dir = os.getcwd() 
    data_folder_path = os.path.join(current_dir, data_folder_name)

    # 确保文件夹存在
    if os.path.exists(data_folder_path):
        supply_chain_graph = build_supply_chain_network(data_folder=data_folder_path)
    else:
        print(f"错误: 找不到数据文件夹 {data_folder_path}")

In [ ]:
import networkx as nx
import pandas as pd
import pickle
import os

def inspect_supply_chain_network(data_folder='data', input_filename='supply_chain_network.pkl'):
    """
    加载 NetworkX 图对象，展示统计信息，并导出节点和边的详细表。
    """
    file_path = os.path.join(data_folder, input_filename)
    
    # 检查文件是否存在
    if not os.path.exists(file_path):
        print(f"错误：找不到文件 {file_path}")
        return

    print(f"--- 正在加载网络图: {file_path} ---")
    with open(file_path, 'rb') as f:
        G = pickle.load(f)

    # --- 1. 基础统计信息 ---
    print("[1] 网络概览 (Network Summary)")
    print("-" * 30)
    print(f"图类型 (Type): {type(G)}")
    print(f"是否有向 (Is Directed): {G.is_directed()}")
    print(f"节点总数 (Total Nodes): {G.number_of_nodes()}")
    print(f"边总数 (Total Edges): {G.number_of_edges()}")
    
    # --- 2. 提取节点详细信息 ---
    print("[2] 处理节点信息 (Nodes)...")
    node_data_list = []
    # 遍历图中所有节点及其属性数据
    for node_id, attributes in G.nodes(data=True):
        row = {'Node_ID': node_id}
        row.update(attributes)
        node_data_list.append(row)
    
    df_nodes = pd.DataFrame(node_data_list)
    
    # 调整DataFrame列顺序，将关键列（ID、名称、位置、SKU）放前面，便于阅读
    priority_cols = ['Node_ID', 'name', 'location', 'sku', 'sku_type', 'is_customer_facing', 'holdcost']
    cols = [c for c in priority_cols if c in df_nodes.columns] + [c for c in df_nodes.columns if c not in priority_cols]
    df_nodes = df_nodes[cols]

    # --- 3. 提取边详细信息 ---
    print("[3] 处理边信息 (Edges)...")
    edge_data_list = []
    # 遍历图中所有边及其属性数据
    for u, v, attributes in G.edges(data=True):
        # 获取源节点和目标节点的名称（比纯数字ID更直观）
        source_name = G.nodes[u].get('name', str(u))
        target_name = G.nodes[v].get('name', str(v))
        
        row = {
            'Source_ID': u,
            'Source_Name': source_name,
            'Target_ID': v,
            'Target_Name': target_name
        }
        row.update(attributes)
        edge_data_list.append(row)
        
    df_edges = pd.DataFrame(edge_data_list)

    # --- 4. 展示和导出 ---
    print("="*50)
    print("节点信息示例 (前 5 行):")
    print("="*50)
    print(df_nodes.head().to_string(index=False))

    print("="*50)
    print("边信息示例 (前 5 行):")
    print("="*50)
    if not df_edges.empty:
        print(df_edges.head().to_string(index=False))
    else:
        print("网络中没有边。")

    # 导出到CSV文件
    output_nodes_csv = os.path.join(data_folder, 'network_nodes_detail.csv')
    output_edges_csv = os.path.join(data_folder, 'network_edges_detail.csv')
    
    df_nodes.to_csv(output_nodes_csv, index=False, encoding='utf-8-sig')
    df_edges.to_csv(output_edges_csv, index=False, encoding='utf-8-sig')

    print("\n" + "="*50)
    print(f"完整信息已导出:")
    print(f"1. 节点明细: {output_nodes_csv}")
    print(f"2. 边明细:   {output_edges_csv}")
    print("="*50)

if __name__ == "__main__":
    current_dir = os.getcwd()
    data_folder_path = os.path.join(current_dir, 'data')

    inspect_supply_chain_network(data_folder=data_folder_path)

In [ ]:
### initial.py ###
import pandas as pd
import numpy as np
import networkx as nx
import pickle
import os
from scipy.stats import norm

def calculate_initial_values(data_folder='.'):
    """
    功能：计算供应链中每个节点的初始库存水位 (Base Stock)。
    核心逻辑：
    1. 需求回溯：从下游（门店）向上游（RDC、工厂）逐级汇总需求均值和波动。
    2. 库存公式：BaseStock = 周转库存 + 安全库存。
    """
    print("--- 开始计算初始库存策略 (基于全链路需求汇总) ---")

    # --- 1. 准备路径与加载数据 ---
    network_path = os.path.join(data_folder, 'supply_chain_network.pkl')
    sales_path = os.path.join(data_folder, '销量分布.csv')
    output_csv_path = os.path.join(data_folder, 'initial.csv')

    # 加载网络图结构
    if not os.path.exists(network_path):
        print(f"错误: 找不到网络文件 {network_path}")
        return
    with open(network_path, 'rb') as f:
        G = pickle.load(f)

    # 加载外部销量数据 (门店/直销客户的直接需求)
    if not os.path.exists(sales_path):
        print(f"错误: 找不到销量文件 {sales_path}")
        return
    df_sales = pd.read_csv(sales_path)
    sales_map = df_sales.set_index(['LocationCode', 'SKUCode']).to_dict('index')

    # --- 2. 定义递归函数：计算总需求 (Direct + Derived) ---
    demand_cache = {}

    def get_aggregated_demand(node_id):
        if node_id in demand_cache:
            return demand_cache[node_id]
        
        node_attrs = G.nodes[node_id]
        loc = node_attrs.get('location')
        sku = node_attrs.get('sku')
        
        # A. 直接需求 (Direct Demand): 仅面向终端消费者的节点有此项
        direct_mean = 0.0
        direct_var = 0.0 
        
        sales_info = sales_map.get((loc, sku))
        if sales_info:
            direct_mean = sales_info['SaleQtyMean']
            direct_var = sales_info['SaleQtyStd'] ** 2
            
        # B. 派生需求 (Derived Demand): 递归汇总所有下游节点的需求
        incoming_mean = 0.0
        incoming_var = 0.0

        for u, v, data in G.in_edges(node_id, data=True):
            child_mean, child_std = get_aggregated_demand(u)
            weight = data.get('weight', 1.0)
            # 均值线性叠加，方差平方叠加 (假设独立)
            incoming_mean += child_mean * weight
            incoming_var += (child_std ** 2) * (weight ** 2)
            
        # C. 汇总并缓存
        total_mean = direct_mean + incoming_mean
        total_std = np.sqrt(direct_var + incoming_var)
        
        demand_cache[node_id] = (total_mean, total_std)
        return total_mean, total_std

    # --- 3. 遍历全网节点计算库存 ---
    initial_data = []
    stats = {'FG': 0, 'RM': 0, 'Zero': 0}

    for node_id in G.nodes():
        attrs = G.nodes[node_id]
        loc = attrs.get('location')
        sku = attrs.get('sku')
        sku_type = attrs.get('sku_type', 'FG')
        
        # [关键步骤] 递归获取该节点的真实总需求
        agg_mean, agg_std = get_aggregated_demand(node_id)

        L = attrs.get('leadtime', 0)
        R = attrs.get('replenishment_cycle', 1)
        target_sl = attrs.get('service_level', 0.95)
        
        base_stock = 0.0
        
        if agg_mean > 0:
            # 计算安全系数 Z (例如 95% -> 1.645)
            z_score = norm.ppf(target_sl)
            
            # --- 标准库存公式 ---
            # Cycle Stock (周转库存) = 覆盖 L+R 期间的平均需求
            cycle_stock = agg_mean * (L + R)
            # Safety Stock (安全库存) = 覆盖 L+R 期间的需求波动
            safety_stock = z_score * agg_std * np.sqrt(L + R)
            
            # 最终库存 = CS + SS * (1.2放大系数)
            base_stock = cycle_stock + safety_stock * 1.2
            base_stock = max(1.0, base_stock)
            
            stats[sku_type] += 1
        else:
            stats['Zero'] += 1

        initial_data.append({
            'Location': loc,
            'SKU': sku,
            'InitialStock': round(base_stock, 0),
            'AggMean': round(agg_mean, 2),
            'Type': sku_type
        })

    # --- 4. 输出结果 ---
    df_init = pd.DataFrame(initial_data)
    df_output = df_init[['Location', 'SKU', 'InitialStock']]
    df_output.to_csv(output_csv_path, index=False)

    print(f"计算完成。成品/半成品: {stats['FG']}, 原材料: {stats['RM']}, 无需求节点: {stats['Zero']}")
    print(f"结果已保存至: {output_csv_path}")

if __name__ == "__main__":
    current_dir = os.getcwd()
    data_folder_path = os.path.join(current_dir, 'data')
    if os.path.exists(data_folder_path):
        calculate_initial_values(data_folder=data_folder_path)
    else:
        print("错误: 找不到数据文件夹")

In [ ]:
### solve.py ###
import numpy as np
import os
import pickle
from rnnisa.model import simulation_lead_real, simu_opt
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

def plot_cost_curve(history, output_folder='.'):
    if not history:
        print("警告：无历史记录，无法绘图。")
        return

    total_costs = [item[0] for item in history]
    holding_costs = [item[1] for item in history]
    epochs = [item[2] for item in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle('Optimization Cost Curves', fontsize=18)

    ax1.plot(epochs, total_costs, marker='.', linestyle='-', color='royalblue', markersize=4)
    ax1.set_title('Avg Total Cost', fontsize=14)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Cost', fontsize=12)
    ax1.grid(True, linestyle='--', linewidth=0.5)

    ax2.plot(epochs, holding_costs, marker='.', linestyle='-', color='darkorange', markersize=4)
    ax2.set_title('Avg Holding Cost', fontsize=14)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Cost', fontsize=12)
    ax2.grid(True, linestyle='--', linewidth=0.5)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plot_path = os.path.join(output_folder, 'cost_curves.png')
    plt.savefig(plot_path)
    plt.close()
    print(f"\n曲线图已保存: {plot_path}")

def load_initial_stock_from_csv(nodes_num, node_map, initial_csv_path):
    """
    辅助函数：从 CSV 加载初始库存向量
    """
    I_S = np.zeros((1, nodes_num), dtype=np.float32)
    
    if not os.path.exists(initial_csv_path):
        print("提示: 未找到 initial.csv，将使用全0初始值。")
        return I_S
        
    try:
        df_init = pd.read_csv(initial_csv_path)
        # 创建查找表: (Location, SKU) -> Value
        init_map = df_init.set_index(['Location', 'SKU'])['InitialStock'].to_dict()
        
        count = 0
        for (loc, sku), node_id in node_map.items():
            if (loc, sku) in init_map:
                val = init_map[(loc, sku)]
                I_S[0, node_id] = val
                if val > 0:
                    count += 1
        print(f"已从 {os.path.basename(initial_csv_path)} 加载初始值，包含 {count} 个非零节点。")
        
    except Exception as e:
        print(f"警告: 读取 initial.csv 失败，回退到全0初始化。错误: {e}")
        
    return I_S

def solve_inventory_optimization(network_file_path, data_folder='.', results_folder='.'):
    """
    主流程：加载模型 -> 加载/计算初值 -> 单阶段优化 -> 输出结果。
    """
    print("--- 开始多级库存优化 ---")

    # --- 1. 加载网络 ---
    with open(network_file_path, 'rb') as f:
        G = pickle.load(f)

    # --- 2. 配置文件路径 ---
    DISTRIBUTION_FILENAME = '销量分布.csv'
    PENALTY_FILENAME = 'penalty_factor.csv'
    INITIAL_FILENAME = 'initial.csv'
    
    # --- 3. 设置优化参数 (单阶段) ---
    DATA_TYPE = np.float32
    SIMULATION_DURATION = 365   
    REP_NUM = 20                
    
    # 优化超参数
    MAX_EPOCHS = 100           
    LEARNING_RATE = 1e-3
    TOLERANCE = 1e1
    PATIENCE = 20
    TARGET_COST = 3.3e9

    print(f"参数配置: Epochs={MAX_EPOCHS}, LR={LEARNING_RATE}, TargetCost={TARGET_COST:.2e}")

    # --- 4. 初始化仿真器 ---
    temp_network_path = os.path.join(data_folder, 'temp_network_for_sim.pkl')
    with open(temp_network_path, 'wb') as f:
        pickle.dump(G, f)

    sim = simulation_lead_real.Simulation(
        data_type=DATA_TYPE,
        data_path=data_folder,
        network_name=os.path.basename(temp_network_path),
        distribution_filename=DISTRIBUTION_FILENAME,
        simulation_duration=SIMULATION_DURATION,
        penalty_filename=PENALTY_FILENAME
    )

    # --- 5. 初始化库存 (使用 initial.csv) ---
    nodes_num = G.number_of_nodes()
    
    # 构建辅助映射: (Loc, SKU) -> NodeID
    node_map_keys = {}
    for node_id, attrs in G.nodes(data=True):
        node_map_keys[(attrs['location'], attrs['sku'])] = node_id
        
    initial_csv_path = os.path.join(data_folder, INITIAL_FILENAME)
    
    # 调用加载函数
    I_S_0 = load_initial_stock_from_csv(nodes_num, node_map_keys, initial_csv_path)

    # --- 6. 执行单阶段优化 ---
    print("--- 开始优化 (SGD) ---")
    optimizer = simu_opt.SimOpt(
        data_path=results_folder,
        rep_num=REP_NUM,
        step_size=LEARNING_RATE,
        positive_flag=True,
        grad_f=sim.evaluate_cost_gradient,
        max_epochs=MAX_EPOCHS,
        convergence_tolerance=TOLERANCE,
        patience=PATIENCE,
        target_cost=TARGET_COST
    )
    
    # 运行优化
    final_policy, history = optimizer.SGD(I_S_0)
    print("--- 优化结束 ---")

    # --- 7. 结果处理 ---
    # 绘制曲线
    plot_cost_curve(history, results_folder)
    
    MIN_STOCK_THRESHOLD = 10.0
    final_policy = np.maximum(final_policy, MIN_STOCK_THRESHOLD)
    
    # 最终评估 (取20次均值)
    final_cost, _, final_holding = sim.evaluate_cost_gradient(final_policy, rep_num=20, mean_flag=True)
    print(f"最终平均总成本: {final_cost:,.2f}")
    print(f"最终平均持有成本: {final_holding:,.2f}")

    # 导出策略 CSV
    node_name_map = {node_id: data['name'] for node_id, data in G.nodes(data=True)}
    results = []
    for idx in range(nodes_num):
        stock_level = np.floor(final_policy[0, idx])
        if stock_level > 0: # 只记录大于0的策略
            node_name = node_name_map.get(idx, f"Node_{idx}")
            try:
                loc, sku = node_name.split('_', 1)
            except ValueError:
                loc, sku = node_name, ""
            results.append({
                'Location': loc, 
                'SKU': sku, 
                'BaseStockLevel': int(stock_level) 
            })
    
    df_results = pd.DataFrame(results).sort_values(by=['Location', 'SKU'])
    output_csv = os.path.join(results_folder, f'result_{len(history)}epochs_{SIMULATION_DURATION}days.csv')
    df_results.to_csv(output_csv, index=False)
    print(f"策略已保存至: {output_csv}")

    # 清理临时文件
    if os.path.exists(temp_network_path):
        os.remove(temp_network_path)

if __name__ == '__main__':
    # 路径处理
    curr_dir = os.getcwd()
    data_dir = os.path.join(curr_dir, 'data')
    net_file = os.path.join(data_dir, 'supply_chain_network.pkl')
    
    solve_inventory_optimization(net_file, data_folder=data_dir, results_folder=curr_dir)

In [ ]:
### service_level_evaluation.py ###
import os
import numpy as np
import pandas as pd
import networkx as nx
import pickle
from warnings import filterwarnings

filterwarnings('ignore')

def calculate_service_level_stochastic(policy_file_path, data_folder='.', rep_num=50, simulation_duration=365):
    """
    功能：对给定的库存策略进行全网络、多轮次的随机仿真评估。
    输出：每个节点的服务水平 (CSL) 和 满足率 (Fill Rate)。
    
    业务逻辑：
    1. 随机生成符合正态分布的外部需求。
    2. 按拓扑顺序，逐天、逐节点推演：
       - 接收上游到货。
       - 满足外部和内部需求 (优先满足内部，剩余满足外部)。
       - 记录缺货情况。
       - 根据 Base Stock 策略向上游发起补货。
    3. 针对成品节点，实施“周期末清零”逻辑 (Lost Sales)。
    """
    print(f"--- 开始全网服务水平评估 (随机模拟模式, 重复 {rep_num} 次) ---")

    # --- 1. 加载数据 ---
    network_file_path = os.path.join(data_folder, 'supply_chain_network.pkl')
    distribution_file_path = os.path.join(data_folder, '销量分布.csv')

    try:
        with open(network_file_path, 'rb') as f:
            G = pickle.load(f)
        df_policy = pd.read_csv(policy_file_path)
        df_dist = pd.read_csv(distribution_file_path)
        print("模型与数据加载成功。")
    except FileNotFoundError as e:
        print(f"错误: 文件未找到 - {e}。")
        return

    # --- 2. 准备基础参数 ---
    nodes_num = G.number_of_nodes()
    data_type = np.float32 
    
    # 提取节点属性：提前期、补货周期、是否面向客户
    lead_times = np.array([int(round(d.get('leadtime', 0))) for _, d in G.nodes(data=True)])
    replenishment_cycles = np.array([G.nodes[i].get('replenishment_cycle', 1) for i in range(nodes_num)], dtype=int)
    replenishment_cycles[replenishment_cycles == 0] = 1
    
    is_customer_facing = np.array([G.nodes[i].get('is_customer_facing', False) for i in range(nodes_num)], dtype=bool)

    # 建立ID与名称的映射
    node_id_to_data = {n: d for n, d in G.nodes(data=True)}
    node_name_to_id = {d['name']: n for n, d in G.nodes(data=True)}
    id_to_node_name = {n: d['name'] for n, d in G.nodes(data=True)}

    # 拓扑排序：确保需求从下游向上游正确传导
    try:
        topo_order = list(nx.topological_sort(G))
    except nx.NetworkXUnfeasible:
        print("警告：网络中检测到循环，退化为按ID顺序计算。")
        topo_order = list(range(nodes_num))

    # --- 3. 解析外部销量分布 ---
    print("正在解析外部销量分布...")
    ext_demand_mean = np.zeros(nodes_num, dtype=data_type)
    ext_demand_std = np.zeros(nodes_num, dtype=data_type)
    
    for row in df_dist.itertuples(index=False):
        node_name = f"{row.LocationCode}_{row.SKUCode}"
        if node_name in node_name_to_id:
            nid = node_name_to_id[node_name]
            ext_demand_mean[nid] = row.SaleQtyMean
            ext_demand_std[nid] = row.SaleQtyStd

    # --- 4. 解析待评估的库存策略 (Base Stock Levels) ---
    I_S = np.zeros(nodes_num, dtype=data_type)
    for row in df_policy.itertuples(index=False):
        node_name = f"{row.Location}_{row.SKU}"
        if node_name in node_name_to_id:
            I_S[node_name_to_id[node_name]] = row.BaseStockLevel

    nodes_to_evaluate = list(range(nodes_num))
    print(f"将在 {nodes_num} 个节点上进行全链路服务水平评估。")

    # --- 5. 初始化统计容器 ---
    global_stats = {
        nid: {
            'total_cycles': 0, 
            'no_stockout_cycles': 0,
            'total_demand_qty': 0.0,
            'fulfilled_qty': 0.0
        }
        for nid in nodes_to_evaluate
    }
    all_stockout_details = []

    # --- 6. 执行多轮模拟 (Monte Carlo Loop) ---
    max_lead_time = int(lead_times.max()) if len(lead_times) > 0 else 0
    
    for run_idx in range(1, rep_num + 1):
        # 6.a 生成全周期的随机需求矩阵 (非负整数)
        random_noise = np.random.normal(loc=ext_demand_mean, scale=ext_demand_std, 
                                        size=(simulation_duration, nodes_num))
        ext_demand_matrix = np.maximum(np.round(random_noise), 0).astype(data_type)

        # 6.b 初始化单次运行的状态变量
        I_on_hand = I_S.copy() 
        pipeline_len = simulation_duration + max_lead_time + 10
        I_pipeline = np.zeros((pipeline_len, nodes_num), dtype=data_type)
        backlog_ext = np.zeros(nodes_num, dtype=data_type)
        cycle_stats = {
            nid: {'demand': 0.0, 'fulfilled': 0.0}
            for nid in nodes_to_evaluate
        }

        # 6.c 逐日推演 (Time Step Loop)
        for t in range(simulation_duration):
            I_on_hand += I_pipeline[t, :]
            daily_internal_reqs = {i: [] for i in range(nodes_num)}
            current_pipeline_sum = np.sum(I_pipeline[t+1:, :], axis=0)
            
            for nid in topo_order:
                # --- 3.1 周期重置逻辑 (Lost Sales) ---
                if t % replenishment_cycles[nid] == 0:
                    if is_customer_facing[nid]:
                        backlog_ext[nid] = 0.0
                
                # --- 3.2 汇总需求 ---
                new_ext_demand = ext_demand_matrix[t, nid]
                backlog_ext[nid] += new_ext_demand
                internal_demand = sum(req[1] for req in daily_internal_reqs[nid])
                total_demand_to_fill = backlog_ext[nid] + internal_demand
                
                # --- 3.3 执行发货/扣减库存 ---
                fulfilled_qty = min(I_on_hand[nid], total_demand_to_fill)
                I_on_hand[nid] -= fulfilled_qty
                
                # --- 3.4 分配给内外需求 (按比例) ---
                fill_rate = 1.0
                if total_demand_to_fill > 1e-6:
                    fill_rate = fulfilled_qty / total_demand_to_fill

                fulfilled_internal = np.round(internal_demand * fill_rate)
                fulfilled_ext = fulfilled_qty - fulfilled_internal

                backlog_ext[nid] -= fulfilled_ext
                if backlog_ext[nid] < 0: backlog_ext[nid] = 0.0
                
                # --- 3.5 记录统计 ---
                cycle_stats[nid]['demand'] += (new_ext_demand + internal_demand)
                cycle_stats[nid]['fulfilled'] += fulfilled_qty
                
                # --- 3.6 向下游发货 (Internal Shipment) ---
                for req_id, req_qty, arrival_idx in daily_internal_reqs[nid]:
                    shipped_qty = np.round(req_qty * fill_rate)
                    if shipped_qty > 0 and arrival_idx < pipeline_len:
                        I_pipeline[arrival_idx, req_id] += shipped_qty
                
                # --- 3.7 向上游订货 (Replenishment) ---
                inv_pos = I_on_hand[nid] + current_pipeline_sum[nid]
                
                order_qty = 0.0
                # 仅在补货日触发
                if t % replenishment_cycles[nid] == 0:
                    raw_order = I_S[nid] - inv_pos
                    order_qty = np.round(max(0, raw_order))
                
                # --- 3.8 订单传导 ---
                if order_qty > 0:
                    sources = list(G.out_edges(nid, data=True))
                    if not sources:
                        # 无上游供应商 (无限产能源头)，直接进入自身 Pipeline
                        lt = lead_times[nid]
                        arr_day = t + lt
                        if arr_day < pipeline_len:
                            I_pipeline[arr_day, nid] += order_qty
                    else:
                        # 有上游，将订单拆分发给供应商
                        for u, v, data in sources:
                            supplier_id = v
                            usage_qty = data.get('weight', 1.0)
                            demand_on_supplier = np.round(order_qty * usage_qty)
                            
                            lt = lead_times[nid] 
                            arr_day = t + lt
                            
                            if demand_on_supplier > 0:
                                daily_internal_reqs[supplier_id].append((nid, demand_on_supplier, arr_day))

            # 周期末结算 (Cycle Accounting)
            for nid in nodes_to_evaluate:
                is_end_of_cycle = (t > 0) and ((t + 1) % replenishment_cycles[nid] == 0)
                is_last_day = (t == simulation_duration - 1)
                
                if is_end_of_cycle or is_last_day:
                    stats = cycle_stats[nid]
                    shortfall = stats['demand'] - stats['fulfilled']

                    global_stats[nid]['total_cycles'] += 1
                    global_stats[nid]['total_demand_qty'] += stats['demand']
                    global_stats[nid]['fulfilled_qty'] += stats['fulfilled']

                    if shortfall < 0.1:
                        global_stats[nid]['no_stockout_cycles'] += 1
                    else:
                        node_name = id_to_node_name[nid]
                        loc, sku = node_name.split('_', 1) if '_' in node_name else (node_name, "")
                        all_stockout_details.append({
                            'Run': run_idx,
                            'Location': loc, 'SKU': sku,
                            'Type': node_id_to_data[nid].get('sku_type', 'Unknown'),
                            'Day': t,
                            'Cycle_Demand': round(stats['demand'], 2),
                            'Cycle_Fulfilled': round(stats['fulfilled'], 2),
                            'Shortfall': round(shortfall, 2)
                        })
                    
                    stats['demand'] = 0.0
                    stats['fulfilled'] = 0.0

    print("多次模拟完成。")

    # --- 7. 计算最终指标并输出 ---
    final_results = []
    for nid in nodes_to_evaluate:
        g_stats = global_stats[nid]
        node_name = id_to_node_name[nid]
        try:
            loc, sku = node_name.split('_', 1)
        except ValueError:
            loc, sku = node_name, ""
        
        sku_type = node_id_to_data[nid].get('sku_type', 'Unknown')
        
        # 计算 CSL (Cycle Service Level) = 不缺货周期数 / 总周期数
        total_cyc = g_stats['total_cycles']
        hits = g_stats['no_stockout_cycles']
        csl = 0.0 if total_cyc == 0 else (hits / total_cyc) * 100.0
        
        # 计算 Fill Rate = 总满足量 / 总需求量
        tot_dem = g_stats['total_demand_qty']
        tot_ful = g_stats['fulfilled_qty']
        fr = 100.0 if tot_dem == 0 else (tot_ful / tot_dem) * 100.0
        
        final_results.append({
            'Location': loc,
            'SKU': sku,
            'Type': sku_type,
            'CSL': round(csl, 2),
            'FillRate': round(fr, 2),
            'Total_Cycles': total_cyc,
            'Stockout_Cycles': total_cyc - hits,
            'Total_Demand': round(tot_dem, 0)
        })

    df_results = pd.DataFrame(final_results).sort_values(by=['Type', 'Location', 'SKU'])
    df_stockouts = pd.DataFrame(all_stockout_details)

    print(f" 全网库存服务水平评估 ({rep_num} Runs) ".center(60, "="))
    for stype in ['FG', 'RM']:
        subset = df_results[df_results['Type'] == stype]
        if not subset.empty:
            print(f"--- {stype} (Top 10 rows) ---")
            print(subset[['Location', 'SKU', 'CSL', 'FillRate', 'Total_Demand']].head(10).to_string(index=False))
            print(f"平均 CSL: {subset['CSL'].mean():.2f}%")

    path_summary = os.path.join(data_folder, 'service_level_summary.csv')
    path_details = os.path.join(data_folder, 'stockout_details.csv')
    
    df_results.to_csv(path_summary, index=False)
    df_stockouts.to_csv(path_details, index=False)
    
    print("="*60)
    print(f"汇总结果已保存至: {path_summary}")
    print(f"缺货明细已保存至: {path_details}")

    return df_results

if __name__ == '__main__':
    base_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
    data_dir = os.path.join(base_dir, 'data')
    policy_file = 'result_100epochs_365days.csv' 
    policy_path = os.path.join(base_dir, policy_file)

    if os.path.exists(policy_path) and os.path.exists(data_dir):
        calculate_service_level_stochastic(
            policy_path, 
            data_folder=data_dir, 
            rep_num=50,       
            simulation_duration=365
        )
    else:
        print(f"错误：找不到文件。请检查路径:\n1. {data_dir}\n2. {policy_path}")

In [ ]:
### adjust.py ###
import pandas as pd
import pickle
import os
import numpy as np

def adjust_penalty_factors(data_folder='.', 
                           sim_result_file='service_level_summary.csv', 
                           penalty_file='penalty_factor.csv'):
    """
    根据仿真结果动态调整惩罚系数。
    规则:
    1. 指标选择: IDC看FillRate, RDC/CDC看CSL。
    2. 调整策略: 差距大则大幅加码(x3.0), 差距小则微调(x1.5), 表现过剩则降本(x0.8)。
    3. 硬性约束: 系数严格限制在 [10, 500000] 之间。
    """
    print("--- 开始动态调整惩罚系数 (带上下限约束) ---")

    # --- 1. 路径检查 ---
    pkl_path = os.path.join(data_folder, 'supply_chain_network.pkl')
    sim_res_path = os.path.join(data_folder, sim_result_file)
    current_penalty_path = os.path.join(data_folder, penalty_file)

    if not (os.path.exists(pkl_path) and os.path.exists(sim_res_path) and os.path.exists(current_penalty_path)):
        print("错误：缺少必要文件，请检查数据文件夹。")
        return

    # --- 2. 数据加载 ---
    # 加载网络图获取目标服务水平 (Target SL)
    with open(pkl_path, 'rb') as f:
        G = pickle.load(f)
    
    target_sl_map = {}
    for _, attrs in G.nodes(data=True):
        target_sl_map[(attrs.get('location'), attrs.get('sku'))] = attrs.get('service_level', 0.95)

    # 加载当前系数和仿真结果
    df_penalty = pd.read_csv(current_penalty_path)
    df_sim = pd.read_csv(sim_res_path)
    sim_data_map = df_sim.set_index(['Location', 'SKU'])[['CSL', 'FillRate']].to_dict('index')
    
    print(f"当前系数表行数: {len(df_penalty)}, 仿真结果节点数: {len(sim_data_map)}")

    # --- 3. 核心调整逻辑 ---
    # 定义硬性上下限
    MIN_PENALTY_LIMIT = 10.0
    MAX_PENALTY_LIMIT = 500000.0
    
    adjusted_data = []
    stats = {'increased': 0, 'decreased': 0, 'unchanged': 0, 'capped_max': 0, 'capped_min': 0}

    for _, row in df_penalty.iterrows():
        loc, sku = row['LocationCode'], row['SKUCode']
        current_factor = row['PenaltyFactor']
        
        # 默认值
        new_factor = current_factor
        metric_name = 'N/A'
        actual_val = np.nan
        target_sl = np.nan
        gap = np.nan
        action = 'N/A'

        # 如果该节点在仿真中有结果
        if (loc, sku) in sim_data_map:
            sim_metrics = sim_data_map[(loc, sku)]
            target_sl = target_sl_map.get((loc, sku), 0.95)
            
            # A. 差异化指标选择
            if 'IDC' in loc:
                actual_val = sim_metrics.get('FillRate', 0) / 100.0
                metric_name = 'FillRate'
            else:
                actual_val = sim_metrics.get('CSL', 0) / 100.0
                metric_name = 'CSL'
            
            gap = target_sl - actual_val
            action = "Kept"
            
            # B. 分段调整倍率
            if gap >= 0.10:     # 严重不达标
                new_factor *= 3.0
                action = "↑ Big Boost"
                stats['increased'] += 1
            elif gap >= 0.05:   # 中度不达标
                new_factor *= 2.0
                action = "↑ Boost"
                stats['increased'] += 1
            elif gap > 0:       # 轻微不达标
                new_factor *= 1.5
                action = "↑ Adjust"
                stats['increased'] += 1
            elif actual_val >= 0.99 and target_sl <= 0.98: # 表现过剩
                new_factor *= 0.8
                action = "↓ Relax"
                stats['decreased'] += 1
            else:
                stats['unchanged'] += 1
        else:
            # 无仿真结果的节点，保持原值，但也需检查上下限
            action = "Init Check"

        # C. 强制上下限截断 (Clamping)
        if new_factor > MAX_PENALTY_LIMIT:
            new_factor = MAX_PENALTY_LIMIT
            action = "Max Capped"
            stats['capped_max'] += 1
        elif new_factor < MIN_PENALTY_LIMIT:
            new_factor = MIN_PENALTY_LIMIT
            action = "Min Capped"
            stats['capped_min'] += 1
            
        # D. 构建新行
        new_row = row.copy()
        new_row['PenaltyFactor'] = round(new_factor, 2)
        new_row.update({
            'MetricUsed': metric_name, 'ActualVal': round(actual_val, 3) if pd.notna(actual_val) else np.nan,
            'TargetSL': target_sl, 'Gap': round(gap, 3) if pd.notna(gap) else np.nan,
            'Action': action
        })
        adjusted_data.append(new_row)

    # --- 4. 保存输出 ---
    df_adjusted = pd.DataFrame(adjusted_data)
    
    # 调整列序方便阅读
    output_cols = ['LocationCode', 'SKUCode', 'PenaltyFactor', 'MetricUsed', 'ActualVal', 'TargetSL', 'Action']
    final_cols = output_cols + [c for c in df_adjusted.columns if c not in output_cols]
    df_adjusted = df_adjusted[final_cols]

    output_path = os.path.join(data_folder, 'penalty_factor.csv')
    df_adjusted.to_csv(output_path, index=False, encoding='utf-8-sig')

    # --- 5. 打印摘要 ---
    print("="*60)
    print("调整完成。统计信息:")
    print(f"  调高: {stats['increased']} | 调低: {stats['decreased']} | 保持: {stats['unchanged']}")
    print(f"  触顶 ({MAX_PENALTY_LIMIT}): {stats['capped_max']} | 触底 ({MIN_PENALTY_LIMIT}): {stats['capped_min']}")
    print("-" * 60)

    print(f"文件已更新至: {output_path}")

if __name__ == '__main__':
    current_dir = os.getcwd()
    data_folder = os.path.join(current_dir, 'data')
    sim_file = 'service_level_summary.csv'
    
    if os.path.exists(os.path.join(data_folder, sim_file)):
        adjust_penalty_factors(data_folder=data_folder, sim_result_file=sim_file)
    else:
        print(f"未找到仿真文件 {sim_file}")